
# Statistical Modeling & Risk-Based Pricing
This notebook covers predictive models that form the core of a dynamic, risk-based pricing system.


In [ ]:

import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
import shap
from modeling import prepare_data_for_modeling, train_and_evaluate_models
import warnings
warnings.filterwarnings('ignore')

# Load data
data = pd.read_csv('../data/insurance_data.csv')
data['Claim_Occurred'] = (data['TotalClaims'] > 0).astype(int)

print("Data loaded. Total shape:", data.shape)


In [ ]:

# Claim Severity Prediction (Risk Model)
# Target: TotalClaims (subset where claims > 0)
severity_data = data[data['TotalClaims'] > 0]
X_train_sev, X_test_sev, y_train_sev, y_test_sev, preprocessor_sev = prepare_data_for_modeling(severity_data, 'TotalClaims')

print("Training severity models...")
results_sev, best_model_sev = train_and_evaluate_models(X_train_sev, X_test_sev, y_train_sev, y_test_sev, task_type='regression')

print("Severity Model Results:")
for name, metrics in results_sev.items():
    print(f"{name}: RMSE={metrics['RMSE']:.2f}, R2={metrics['R2']:.2f}")


In [ ]:

# Claim Probability Prediction
# Target: Claim_Occurred
X_train_cls, X_test_cls, y_train_cls, y_test_cls, preprocessor_cls = prepare_data_for_modeling(data, 'Claim_Occurred')

print("Training probability models...")
results_cls, best_model_cls = train_and_evaluate_models(X_train_cls, X_test_cls, y_train_cls, y_test_cls, task_type='classification')

print("Probability Model Results:")
for name, metrics in results_cls.items():
    print(f"{name}: Accuracy={metrics['Accuracy']:.2f}, F1={metrics['F1']:.2f}")


In [ ]:

# SHAP explanation for the best severity model
# Because SHAP can be slow, we take a small sample
sample_X = shap.sample(X_train_sev, 100)
explainer = shap.Explainer(best_model_sev.predict, sample_X)
shap_values = explainer(sample_X)

shap.plots.bar(shap_values)
